# Lesson 7 — QAOA and Combinatorial Optimization

**Quantum Computing with Qiskit II**

This notebook accompanies Lesson 7. Work through the cells in order.

**Topics covered:**
- Encoding Max-Cut as a cost Hamiltonian
- Building and visualizing the 4-node cycle graph
- QAOA circuit: cost layer, mixer layer, and full circuit
- Optimizing QAOA parameters with COBYLA
- Visualizing the $p=1$ parameter landscape
- Comparing approximation ratios across depths $p = 1, 2, 3$
- Sampling optimal partitions from the QAOA state

In [ ]:
!pip install qiskit qiskit-aer scipy networkx pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer

print("Qiskit:    ", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

print("Imports complete.")

## 1. The Graph: 4-Node Cycle $C_4$

The running example is the 4-node cycle graph $C_4$ with vertices $\{0, 1, 2, 3\}$ and edges $\{(0,1), (1,2), (2,3), (3,0)\}$.

The maximum cut is **4** (all edges can be cut): the bipartite partition $\{0, 2\}$ vs $\{1, 3\}$ places alternating vertices on opposite sides, and every edge crosses the partition.

In [ ]:
import networkx as nx

# Define the graph
n     = 4
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]   # 4-cycle C4

G = nx.Graph()
G.add_nodes_from(range(n))
G.add_edges_from(edges)

# Draw the graph with the bipartite layout to make the max-cut structure clear
pos = {0: (0, 1), 1: (1, 0), 2: (0, -1), 3: (-1, 0)}
node_colors = ['steelblue', 'tomato', 'steelblue', 'tomato']  # {0,2} vs {1,3}

fig, ax = plt.subplots(figsize=(4, 4))
nx.draw_networkx(G, pos=pos, ax=ax,
                 node_color=node_colors, node_size=700,
                 font_color='white', font_size=14,
                 edge_color='gray', width=2, with_labels=True)
ax.set_title("$C_4$: blue vs red = max-cut partition", fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()
# Expected: a square graph with nodes 0, 2 colored blue and 1, 3 colored red.
# Every edge connects a blue node to a red node -- all 4 edges are cut.

## 2. Brute-Force Max-Cut

For a small graph we can check all $2^n$ partitions exhaustively. Each partition is represented as a bitmask: bit $i$ of the mask is 1 if vertex $i$ is in $\bar{S}$.

In [ ]:
def brute_force_max_cut(n, edges):
    # Check all 2^n partitions and return (max_cut_value, list_of_optimal_masks).
    max_cut = 0
    best_masks = []
    for mask in range(2**n):
        cut = sum(1 for i, j in edges
                  if ((mask >> i) & 1) != ((mask >> j) & 1))
        if cut > max_cut:
            max_cut = cut
            best_masks = [mask]
        elif cut == max_cut:
            best_masks.append(mask)
    return max_cut, best_masks


max_cut, best_masks = brute_force_max_cut(n, edges)

print(f"Max-Cut value:       {max_cut}")
print(f"Optimal partitions:")
for mask in best_masks:
    partition = [i for i in range(n) if (mask >> i) & 1]
    complement = [i for i in range(n) if not (mask >> i) & 1]
    print(f"  S = {set(partition)}, S_bar = {set(complement)}, bitstring = {bin(mask)}")
# Expected:
# Max-Cut value: 4
# Two optimal partitions: {0,2} vs {1,3} (masks 0b0101=5 and 0b1010=10)

## 3. Cost Hamiltonian

The Max-Cut objective maps to the cost operator:

$$H_C = \sum_{(i,j) \in E} \frac{1 - Z_i Z_j}{2} = 2\cdot IIII - \frac{1}{2}(IIZZ + IZZI + ZZII + ZIIZ)$$

The maximum eigenvalue of $H_C$ equals the maximum cut value (4 for $C_4$).

In [ ]:
def build_cost_hamiltonian(n, edges):
    # H_C = sum_{(i,j)} (I - Z_i Z_j) / 2
    # Constant term: |E|/2 * IIII
    # ZZ terms: -1/2 * Z_i Z_j for each edge
    pauli_terms = ['I' * n]         # constant term
    coeffs      = [len(edges) / 2]  # |E| / 2
    for i, j in edges:
        s = ['I'] * n
        s[n - 1 - i] = 'Z'          # rightmost char = qubit 0
        s[n - 1 - j] = 'Z'
        pauli_terms.append(''.join(s))
        coeffs.append(-0.5)
    return SparsePauliOp(pauli_terms, coeffs=coeffs)


H_C = build_cost_hamiltonian(n, edges)

print("Cost Hamiltonian H_C:")
for term, coeff in zip(H_C.paulis.to_labels(), H_C.coeffs):
    print(f"  {coeff.real:+.2f} * {term}")

# Verify the maximum eigenvalue equals the max cut
H_matrix = H_C.to_matrix()
evals = np.linalg.eigvalsh(H_matrix)
print(f"\nMax eigenvalue of H_C: {evals[-1]:.4f}  (expected: {max_cut})")
print(f"Min eigenvalue of H_C: {evals[0]:.4f}   (expected: 0.0)")
# Expected:
# H_C terms: +2.00 * IIII, -0.50 * IIZZ, -0.50 * IZZI, -0.50 * ZZII, -0.50 * ZIIZ
# Max eigenvalue: 4.0  (max cut)
# Min eigenvalue: 0.0  (all vertices in same partition)

In [ ]:
# Verify: H_C gives the correct cut value for each computational basis state
print("Cut values for all 16 basis states:")
for mask in range(2**n):
    state = np.zeros(2**n, dtype=complex)
    state[mask] = 1.0  # computational basis state |mask>
    cut_val = (state.conj() @ H_matrix @ state).real
    cut_bf  = sum(1 for i, j in edges if ((mask >> i) & 1) != ((mask >> j) & 1))
    bit_str = format(mask, f'0{n}b')
    ok_str = "OK" if abs(cut_val - cut_bf) < 1e-9 else "MISMATCH"
    print(f"  |{bit_str}> -> H_C eigenvalue = {cut_val:.1f},  brute-force cut = {cut_bf}  {ok_str}")
# Expected: H_C eigenvalue matches brute-force cut for all 16 states.
# |1010> and |0101> should show eigenvalue 4.0 (all edges cut).
# |0000> and |1111> should show eigenvalue 0.0 (no edges cut).

## 4. QAOA Circuit

The QAOA circuit at depth $p$ consists of:
1. Hadamard gates on all qubits: prepares the uniform superposition $|+\rangle^{\otimes n}$.
2. Alternating layers of the cost unitary $U_C(\gamma_l) = e^{-i\gamma_l H_C}$ and the mixer unitary $U_B(\beta_l) = e^{-i\beta_l H_B}$ where $H_B = \sum_i X_i$.

**Cost layer per edge $(i,j)$:** $e^{i(\gamma/2)Z_iZ_j} = \mathrm{CX}(i,j)\cdot R_z(-\gamma,j)\cdot\mathrm{CX}(i,j)$

**Mixer layer per qubit $i$:** $e^{-i\beta X_i} = R_x(2\beta, i)$

In [ ]:
def cost_layer(qc, gamma, edges):
    # Apply e^{i(gamma/2) Z_i Z_j} for each edge (global phase dropped).
    # Circuit identity: CX(i,j) - Rz(-gamma, j) - CX(i,j) = e^{i(gamma/2) ZiZj}
    for i, j in edges:
        qc.cx(i, j)
        qc.rz(-gamma, j)
        qc.cx(i, j)


def mixer_layer(qc, beta, n):
    # Apply e^{-i beta X_i} = Rx(2 beta) to each qubit.
    for i in range(n):
        qc.rx(2 * beta, i)


def qaoa_circuit(n, edges, gamma, beta):
    # Build the depth-p QAOA circuit for Max-Cut.
    # gamma, beta: lists of length p (one value per layer).
    p  = len(gamma)
    qc = QuantumCircuit(n)
    qc.h(range(n))                       # uniform superposition
    for layer in range(p):
        cost_layer(qc, gamma[layer], edges)
        mixer_layer(qc, beta[layer], n)
    return qc


# Inspect the p=1 circuit
qc_p1_example = qaoa_circuit(n, edges, gamma=[0.5], beta=[0.3])
print("QAOA p=1 circuit (example angles gamma=0.5, beta=0.3):")
print(f"  Gate counts: {dict(qc_p1_example.count_ops())}")
print(f"  Depth:       {qc_p1_example.depth()}")
# Expected: {'h': 4, 'cx': 8, 'rz': 4, 'rx': 4}
# 4 Hadamards (init) + 4 edges x 2 CX + 4 Rz (cost layer) + 4 Rx (mixer layer)
# Depth: 8 (Hadamards parallelized; CX pairs partially parallelized)

In [ ]:
qaoa_circuit(n, edges, gamma=[0.5], beta=[0.3]).draw('mpl')

## 5. QAOA Optimization at $p = 1$

We maximize $F_1(\gamma, \beta) = \langle \psi(\gamma, \beta) | H_C | \psi(\gamma, \beta) \rangle$ by minimizing its negative with COBYLA.

The search space is bounded: $\gamma \in [0, 2\pi]$ and $\beta \in [0, \pi]$.

In [ ]:
def qaoa_objective(params, n, edges, H_C, p):
    # Returns -<H_C> (negative because scipy minimizes).
    gamma = params[:p]
    beta  = params[p:]
    qc    = qaoa_circuit(n, edges, gamma, beta)
    sv    = Statevector(qc)
    return -sv.expectation_value(H_C).real


# p = 1 optimization
p = 1
history_p1 = []

def cost_p1(params):
    val = qaoa_objective(params, n, edges, H_C, p)
    history_p1.append(-val)  # store positive <H_C>
    return val


# Grid search over the p=1 landscape to find a good starting point
best_val  = -np.inf
best_init = None
for g0 in np.linspace(0, 2 * np.pi, 10):
    for b0 in np.linspace(0, np.pi, 10):
        val = -qaoa_objective([g0, b0], n, edges, H_C, p)
        if val > best_val:
            best_val  = val
            best_init = [g0, b0]

print(f"Grid search best F1 = {best_val:.4f} at gamma={best_init[0]:.3f}, beta={best_init[1]:.3f}")

result_p1 = minimize(cost_p1, best_init, method='COBYLA',
                     options={'maxiter': 1000, 'rhobeg': 0.1})

gamma_opt_p1 = result_p1.x[:p]
beta_opt_p1  = result_p1.x[p:]
F1_opt       = -result_p1.fun

print(f"\nOptimized p=1 QAOA:")
print(f"  gamma* = {gamma_opt_p1[0]:.4f} rad")
print(f"  beta*  = {beta_opt_p1[0]:.4f} rad")
print(f"  F1*    = {F1_opt:.6f}")
print(f"  Approximation ratio r1 = {F1_opt / max_cut:.4f}")
# Expected:
# F1* ≈ 3.0 (expected cut value = 3 out of maximum 4)
# Approximation ratio r1 ≈ 0.75
# Optimal angles are near gamma* ≈ pi/4 and beta* ≈ pi/8 (up to symmetry)

## 6. The $p = 1$ Parameter Landscape

For $p = 1$, the QAOA objective $F_1(\gamma, \beta)$ is a 2D function that can be visualized as a heat map over $\gamma \in [0, 2\pi]$ and $\beta \in [0, \pi]$.

In [ ]:
# Evaluate F1 on a 40x40 grid
g_vals = np.linspace(0, 2 * np.pi, 40)
b_vals = np.linspace(0, np.pi, 40)
F1_grid = np.zeros((len(b_vals), len(g_vals)))

for bi, b in enumerate(b_vals):
    for gi, g in enumerate(g_vals):
        F1_grid[bi, gi] = -qaoa_objective([g, b], n, edges, H_C, p=1)

fig, ax = plt.subplots(figsize=(7, 4))
cf = ax.contourf(g_vals, b_vals, F1_grid, levels=30, cmap='viridis')
plt.colorbar(cf, ax=ax, label='$F_1(\\gamma, \\beta)$')
ax.plot(gamma_opt_p1[0], beta_opt_p1[0], 'r*', markersize=14,
        label=f'Optimum $F_1^*={F1_opt:.2f}$')
ax.set_xlabel('$\\gamma$', fontsize=13)
ax.set_ylabel('$\\beta$', fontsize=13)
ax.set_title('QAOA $p=1$ landscape for $C_4$', fontsize=12)
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(['0', '$\\pi/2$', '$\\pi$', '$3\\pi/2$', '$2\\pi$'])
ax.set_yticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
ax.set_yticklabels(['0', '$\\pi/4$', '$\\pi/2$', '$3\\pi/4$', '$\\pi$'])
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
# Expected: a smooth landscape with multiple equivalent bright peaks (F1 ≈ 3.0)
# separated by dark troughs (F1 ≈ 1.0--2.0). The red star marks one of the optima.
# The landscape repeats with period 2pi in gamma and pi in beta.

## 7. Sampling the Optimized Circuit

After finding optimal parameters, we compute the probability of each computational basis state in the QAOA output state. High-cut partitions should have the highest probabilities.

In [ ]:
# Compute state probabilities at optimal p=1 parameters
qc_opt = qaoa_circuit(n, edges, gamma_opt_p1, beta_opt_p1)
sv_opt = Statevector(qc_opt)
probs  = sv_opt.probabilities_dict()  # bitstring -> probability

# Compute the cut value for each bitstring
def cut_value(bitstring, edges):
    # bitstring: string of 0/1 with rightmost = qubit 0 (Qiskit convention)
    bits = [int(b) for b in reversed(bitstring)]  # bits[i] = qubit i
    return sum(1 for i, j in edges if bits[i] != bits[j])

# Sort by probability descending
sorted_probs = sorted(probs.items(), key=lambda x: -x[1])

print(f"Top 8 bit strings by probability (p=1 optimum):")
print(f"  {"Bitstring":>10}  {"Probability":>12}  {"Cut value":>10}")
print("-" * 38)
for bs, prob in sorted_probs[:8]:
    cv = cut_value(bs, edges)
    print(f"  {bs:>10}  {prob:12.4f}  {cv:10d}")

# Expected: '1010' and '0101' (the max-cut partitions) have the highest probabilities.
# Their combined probability represents the chance of finding the optimal partition
# in a single shot from the p=1 QAOA circuit.

In [ ]:
# Bar chart of probabilities
labels = [bs for bs, _ in sorted(probs.items())]
values = [probs[bs] for bs in labels]
colors = ['tomato' if cut_value(bs, edges) == max_cut else 'steelblue'
          for bs in labels]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(labels, values, color=colors, edgecolor='gray', linewidth=0.5)
ax.set_xlabel("Bit string (qubit 3 ... qubit 0)", fontsize=11)
ax.set_ylabel("Probability", fontsize=11)
ax.set_title(
    f'QAOA $p=1$ output distribution for $C_4$ '
    f'($\\gamma^*={gamma_opt_p1[0]:.3f}$, $\\beta^*={beta_opt_p1[0]:.3f}$)',
    fontsize=11)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()
# Expected: the two optimal partitions '1010' and '0101' (red bars) have
# the highest probabilities. The remaining 14 bit strings (blue bars) are
# suboptimal and share the remaining probability mass.

## 8. Approximation Ratio vs Circuit Depth

Increasing $p$ allows the QAOA circuit to explore more of the Hilbert space and achieve higher expected cut values. We compare $p = 1, 2, 3$ on $C_4$.

For $p \geq 2$, we use **warm starting**: initialize each new set of angles by linearly interpolating the previously optimized $p-1$ solution.

In [ ]:
def warm_start_params(prev_gamma, prev_beta):
    # Linearly interpolate p-1 parameters to p parameters.
    # Insert a new (gamma, beta) midpoint between each consecutive pair.
    p_prev = len(prev_gamma)
    if p_prev == 0:
        return np.array([np.pi / 4]), np.array([np.pi / 8])
    # New gamma: insert midpoints + final value
    new_gamma = np.interp(np.linspace(0, p_prev - 1, p_prev + 1),
                          np.arange(p_prev), prev_gamma)
    new_beta  = np.interp(np.linspace(0, p_prev - 1, p_prev + 1),
                          np.arange(p_prev), prev_beta)
    return new_gamma, new_beta


depth_results = {}
gamma_prev, beta_prev = np.array([]), np.array([])

print(f"{"p":>3}  {"F_p*":>8}  {"r_p = F_p/OPT":>16}  {"Circuit depth":>14}  {"nfev":>6}")
print("-" * 54)

for p in [1, 2, 3]:
    # Warm-start initialization
    g0, b0 = warm_start_params(gamma_prev, beta_prev)
    init   = np.concatenate([g0, b0])

    res = minimize(lambda params: qaoa_objective(params, n, edges, H_C, p),
                   init, method='COBYLA',
                   options={'maxiter': 3000, 'rhobeg': 0.15})

    F_opt = -res.fun
    r_p   = F_opt / max_cut
    gamma_prev = res.x[:p]
    beta_prev  = res.x[p:]

    qc_depth = qaoa_circuit(n, edges, gamma_prev, beta_prev).depth()
    depth_results[p] = {'F': F_opt, 'r': r_p, 'gamma': gamma_prev, 'beta': beta_prev}

    print(f"  {p}  {F_opt:8.5f}  {r_p:16.5f}  {qc_depth:14d}  {res.nfev:6d}")

# Expected (approximate):
# p=1: F1* ≈ 3.0,   r1 ≈ 0.75,  depth ≈  8
# p=2: F2* ≈ 3.5+,  r2 ≈ 0.87+, depth ≈ 16
# p=3: F3* ≈ 3.8+,  r3 ≈ 0.95+, depth ≈ 24
# Approximation ratio improves with p, approaching 1.0 as p -> infinity.

In [ ]:
# Plot approximation ratio vs depth p
p_vals = sorted(depth_results)
r_vals = [depth_results[p]['r'] for p in p_vals]

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(p_vals, r_vals, 'o-', color='steelblue', linewidth=2, markersize=9,
        label='QAOA $r_p$')
ax.axhline(0.8786, color='tomato', linestyle='--', linewidth=1.5,
           label='Goemans-Williamson (0.8786)')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1.5,
           label='Random partition (0.5)')
ax.set_xlabel('QAOA depth $p$', fontsize=12)
ax.set_ylabel('Approximation ratio $r_p = F_p^*/\\mathrm{OPT}$', fontsize=11)
ax.set_title('QAOA approximation ratio vs depth ($C_4$)', fontsize=12)
ax.set_xticks(p_vals)
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
# Expected: QAOA points rise from ~0.75 at p=1 toward 1.0 as p increases.
# The Goemans-Williamson dashed line at 0.8786 may be crossed at p=2 or p=3
# for this specific graph C4 (which is bipartite and easier than general graphs).

## 9. Sampling and Post-Processing

In practice, the QAOA circuit is run many times (shots) and the best observed cut among all shots is kept. Here we simulate this by drawing from the state probabilities and tracking the best cut found.

In [ ]:
# Simulate 200 shots from the p=3 QAOA circuit and keep the best cut
p_best = max(depth_results)
gamma_best = depth_results[p_best]['gamma']
beta_best  = depth_results[p_best]['beta']

qc_best  = qaoa_circuit(n, edges, gamma_best, beta_best)
sv_best  = Statevector(qc_best)
probs_best = sv_best.probabilities()   # array of length 2^n, index = bitstring

rng   = np.random.default_rng(42)
shots = 200
# Draw sample indices from the probability distribution
samples = rng.choice(2**n, size=shots, p=probs_best)

best_found = 0
cut_history = []
for s in samples:
    bs  = format(s, f'0{n}b')  # e.g. '1010' (qubit 3 leftmost)
    cv  = cut_value(bs, edges)
    best_found = max(best_found, cv)
    cut_history.append(best_found)

print(f"Best cut found in {shots} shots: {best_found}  (optimal = {max_cut})")

fig, ax = plt.subplots(figsize=(7, 3))
ax.step(range(shots), cut_history, color='steelblue', linewidth=1.5)
ax.axhline(max_cut, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Optimal cut = {max_cut}')
ax.set_xlabel("Shot number", fontsize=11)
ax.set_ylabel("Best cut seen so far", fontsize=11)
ax.set_title(f'Best cut over {shots} shots from QAOA $p={p_best}$', fontsize=11)
ax.legend(fontsize=10)
ax.set_yticks(range(max_cut + 2))
plt.tight_layout()
plt.show()
# Expected: the best-cut curve reaches the optimal value (4) quickly,
# typically within the first 10--30 shots for p=3 on C4.
# The exact shot count varies due to the random seed.

## Exercises

Work through each exercise in the cell below it.

### Exercise 1: Different Graph

Build a 5-node graph with edges $(0,1), (1,2), (2,3), (3,4), (4,0), (0,2)$ (a 5-cycle with one extra diagonal edge).

1. Find the maximum cut by brute force.
2. Build the corresponding `SparsePauliOp` cost Hamiltonian for $n = 5$.
3. Run QAOA at $p = 1$ and $p = 2$ and report the approximation ratios.

*Expected: the brute-force max cut should be 5. QAOA $p=1$ will have a lower approximation ratio than for $C_4$ (5-node odd cycle is harder).*

In [ ]:
# Exercise 1: Different graph

n5    = 5
edges5 = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)]

# TODO: brute-force max cut for the 5-node graph
# max5, masks5 = brute_force_max_cut(n5, edges5)
# print(f'Max cut: {max5}')

# TODO: build the SparsePauliOp for n=5
# H_C5 = build_cost_hamiltonian(n5, edges5)

# TODO: run QAOA p=1 and p=2
# Use the same qaoa_circuit, cost_layer, mixer_layer functions -- they are graph-agnostic.

### Exercise 2: Parameter Shift Gradients for QAOA

Apply the parameter shift rule from Lesson 6 to the QAOA objective.

1. For $p=1$ at the initial guess $(\gamma_0, \beta_0) = (0.5, 0.3)$, compute the gradient $\nabla_{\gamma, \beta} F_1$ using the parameter shift rule.
2. Compare to the finite-difference gradient (step size $10^{-5}$).
3. Use the parameter shift gradient in a simple gradient ascent loop and compare convergence to COBYLA.

*Expected: parameter shift and finite difference gradients agree to within $10^{-4}$. Gradient ascent converges to the same optimum as COBYLA but may need more iterations.*

In [ ]:
# Exercise 2: Parameter shift gradients for QAOA

# TODO: implement the parameter shift gradient for qaoa_objective
#
# def qaoa_gradient(params, n, edges, H_C, p):
#     # Gradient of -<H_C> (consistent with minimization convention)
#     grad = np.zeros(len(params))
#     shift = np.pi / 2
#     for k in range(len(params)):
#         e_k      = np.zeros(len(params)); e_k[k] = 1.0
#         grad[k]  = 0.5 * (qaoa_objective(params + shift * e_k, n, edges, H_C, p)
#                          - qaoa_objective(params - shift * e_k, n, edges, H_C, p))
#     return grad

# TODO: compare to finite differences and run gradient ascent

### Exercise 3: Weighted Max-Cut

Extend the cost Hamiltonian to a **weighted graph** where each edge $(i,j)$ has a weight $w_{ij} > 0$:

$$H_C^{(w)} = \sum_{(i,j) \in E} w_{ij} \frac{1 - Z_i Z_j}{2}$$

Use the 4-cycle $C_4$ with weights $w_{01} = 2, w_{12} = 1, w_{23} = 3, w_{30} = 1$.

1. Build the weighted `SparsePauliOp`.
2. Find the optimal weighted cut by brute force.
3. Run QAOA $p=1$ and $p=2$ and report the approximation ratios against the weighted optimum.

*Expected: the brute-force optimum is partition $\{0, 2\}$ vs $\{1, 3\}$ with weighted cut $2 + 1 + 3 + 1 = 7$.*

In [ ]:
# Exercise 3: Weighted Max-Cut

weights = {(0, 1): 2.0, (1, 2): 1.0, (2, 3): 3.0, (3, 0): 1.0}

# TODO: build the weighted cost Hamiltonian
# H_C_w = SparsePauliOp with -w_ij/2 coefficients for each ZZ term
# (and a constant term sum(w_ij)/2 * IIII)

# TODO: brute-force weighted max cut
# def brute_force_weighted(n, weights_dict):
#     max_cut = 0
#     for mask in range(2**n):
#         cut = sum(w for (i, j), w in weights_dict.items()
#                   if ((mask >> i) & 1) != ((mask >> j) & 1))
#         max_cut = max(max_cut, cut)
#     return max_cut

# TODO: run QAOA p=1 and p=2 with H_C_w

### Exercise 4: QAOA vs Classical Greedy

Implement a simple **greedy Max-Cut heuristic**: process vertices one at a time and assign each to the partition that maximizes the current cut value.

1. Implement the greedy algorithm for the 4-cycle $C_4$.
2. Run the greedy algorithm 16 times with different vertex orderings (all permutations of 4 vertices), record the cut value for each, and compute the average.
3. Compare the best greedy cut, average greedy cut, and QAOA $p=1$ expected cut value.

*Expected: greedy gives max cut = 4 on $C_4$ for some orderings (since $C_4$ is bipartite), but fails for others. QAOA $p=1$ gives expected cut ≈ 3.0, lying between the greedy best and worst.*

In [ ]:
# Exercise 4: Greedy Max-Cut vs QAOA

from itertools import permutations

# TODO: implement greedy Max-Cut
#
# def greedy_max_cut(n, edges, order):
#     # Assign vertices in 'order' to the partition that maximizes the current cut.
#     partition = {}  # vertex -> 0 or 1
#     for v in order:
#         cut_if_0 = sum(1 for i, j in edges
#                        if (v == i and j in partition and partition[j] != 0)
#                        or (v == j and i in partition and partition[i] != 0))
#         cut_if_1 = sum(1 for i, j in edges
#                        if (v == i and j in partition and partition[j] != 1)
#                        or (v == j and i in partition and partition[i] != 1))
#         partition[v] = 0 if cut_if_0 >= cut_if_1 else 1
#     mask = sum((partition[v] << v) for v in range(n))
#     return sum(1 for i, j in edges if ((mask >> i) & 1) != ((mask >> j) & 1))

# TODO: run over all permutations(range(4)) and compare to QAOA F1*